In [ ]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [7]:
data = np.load('kanji_data.npz', allow_pickle=True)
X = data['images']
y = data['labels']
class_list = data['class_list']
print(f"Label {y[0]} -> {class_list[y[0]]} -> {chr(int(class_list[y[0]], 16))}")

Label 0 -> 0x3042 -> あ


In [10]:
from torch.utils.data import Dataset, DataLoader

# batch & normalizing

class KanjiDataset(Dataset):
    def __init__(self, image_file, label_file, indices):
        self.images = np.load(image_file, mmap_mode="r")
        self.labels = np.load(label_file, mmap_mode="r")
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        # load an image
        image = self.images[real_idx]
        label = self.labels[real_idx]

        # normalize
        image = image.astype(np.float32) / 255.0

        # add channel dimension: (127,128) -> (1,127,128)
        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(label).long()

        return image, label

In [11]:
from sklearn.model_selection import train_test_split

# train test split

indices = np.arange(len(y))

train_idx, temp_idx = train_test_split(indices, test_size=0.2, stratify=y, random_state=42)

val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y[temp_idx], random_state=42)

In [12]:
train_data = KanjiDataset("kanji_images.npy", "kanji_labels.npy", train_idx)

val_data = KanjiDataset("kanji_images.npy", "kanji_labels.npy", val_idx)

test_data = KanjiDataset( "kanji_images.npy", "kanji_labels.npy", test_idx)


train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [13]:
image, label = train_data[0]
print(image.size())
print(len(class_list))

torch.Size([1, 127, 128])
3036


In [14]:
class NeuralNet(nn.Module):

    def __init__ (self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 12, 5) # (12, 123, 124)
        self.pool = nn.MaxPool2d(2, 2) # (12, 61, 62)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 57, 58) -> apply pool -> (24, 28, 29) -> flatten (24 * 28 * 29)

        self.fc1 = nn.Linear(24 * 28 * 29, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3036) # 3036 kanji/character classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(x.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
for epoch in range(30):
    print(f'Training epoch {epoch}...')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')

Training epoch 0...


c:\Users\johan\Documents\Projects\kanji-reader\kanji\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# export model parameters
torch.save(net.state_dict(), 'kanji_model.pth')

In [ ]:
net = NeuralNet().to(device)
net.load_state_dict(torch.load('kanji_model.pth'))

In [ ]:
correct = 0
total = 0

net.eval()

# feed data in to get predictions and compare to labels
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1) # highest activation becomes the label
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy: {accuracy}%')